# Suraksha - Feature Engineering Script

**Engineers behavioral features using Spark window functions**

Person 1's Third Task - Run after 02_delta_ingestion notebook

## Features Created:
- Velocity features (txn counts in time windows)
- Temporal features (time since last txn)
- Historical features (sender-receiver history)
- Mule account features (inbound/outbound counts)
- Statistical features (z-scores, percentiles)

**Input:** workspace.bronze.upi (raw data)  
**Output:** workspace.silver.upi_features (with behavioral features)

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import (
    col, count, sum as spark_sum, avg, stddev, lag, unix_timestamp, 
    when, lit, datediff, current_date, row_number, percent_rank, countDistinct
)
from pyspark.sql.types import LongType, DoubleType, BooleanType
import time

print("="*70)
print("🚀 SURAKSHA - FEATURE ENGINEERING")
print("="*70)

In [0]:
print("\n📊 Step 1: Reading bronze layer data...")
print("   Source: workspace.bronze.upi")

try:
    df = spark.read.table("workspace.bronze.upi")
    print(f"✓ Loaded {df.count():,} transactions from Delta Lake")
except Exception as e:
    print(f"⚠️  Could not read from Unity Catalog, trying DBFS...")
    df = spark.read.format("delta").load("dbfs:/suraksha/bronze/upi")
    print(f"✓ Loaded {df.count():,} transactions from DBFS")

print(f"✓ Initial schema: {len(df.columns)} columns")

# Convert timestamp to proper format if needed
df = df.withColumn("timestamp", col("timestamp").cast("timestamp"))

In [0]:
print("\n📊 Step 2: Engineering velocity features...")
print("   Using Spark window functions for time-based aggregations")

# Convert timestamp to Unix seconds for calculations FIRST
df = df.withColumn("timestamp_unix", unix_timestamp("timestamp"))

# Define time windows for sender using timestamp_unix (Unix seconds)
sender_window_1min = Window.partitionBy("sender_id").orderBy("timestamp_unix").rangeBetween(-60, 0)
sender_window_1hour = Window.partitionBy("sender_id").orderBy("timestamp_unix").rangeBetween(-3600, 0)
sender_window_24h = Window.partitionBy("sender_id").orderBy("timestamp_unix").rangeBetween(-86400, 0)

# Sender transaction counts in time windows
df = df.withColumn(
    "sender_txn_count_1min",
    count("*").over(sender_window_1min) - 1  # Exclude current transaction
)

df = df.withColumn(
    "sender_txn_count_1hour",
    count("*").over(sender_window_1hour) - 1
)

df = df.withColumn(
    "sender_txn_count_24h",
    count("*").over(sender_window_24h) - 1
)

print("   ✓ Created sender_txn_count_1min, sender_txn_count_1hour, sender_txn_count_24h")

In [0]:
print("\n📊 Step 3: Calculating time since last transaction...")

sender_window_by_time = Window.partitionBy("sender_id").orderBy("timestamp_unix")

df = df.withColumn(
    "prev_txn_time",
    lag("timestamp_unix", 1).over(sender_window_by_time)
)

df = df.withColumn(
    "time_since_last_txn_sec",
    when(col("prev_txn_time").isNotNull(), 
         col("timestamp_unix") - col("prev_txn_time"))
    .otherwise(999999)  # Very large number for first transaction
).drop("prev_txn_time")

print("   ✓ Created time_since_last_txn_sec")

In [0]:
print("\n📊 Step 4: Checking sender-receiver transaction history...")

# Create a window to check if sender and receiver have transacted before
sender_receiver_window = Window.partitionBy("sender_id", "receiver_id").orderBy("timestamp_unix")

df = df.withColumn(
    "txn_sequence",
    row_number().over(sender_receiver_window)
)

df = df.withColumn(
    "sender_receiver_history",
    when(col("txn_sequence") > 1, True).otherwise(False)
).drop("txn_sequence")

print("   ✓ Created sender_receiver_history")

In [0]:
print("\n📊 Step 5: Engineering mule account detection features...")

# Receiver inbound/outbound counts in 1 hour - using timestamp_unix for range windows
receiver_window_1hour = Window.partitionBy("receiver_id").orderBy("timestamp_unix").rangeBetween(-3600, 0)

# Count inbound transactions (receiver perspective)
df = df.withColumn(
    "receiver_inbound_count_1h",
    count("*").over(receiver_window_1hour) - 1
)

# FIX 7: Correct receiver_outbound_count_1h calculation
# To get receiver's outbound: look at transactions where receiver_id appears as sender_id in the same time window
# Step 1: build a lookup of sender activity
sender_activity = df.select(
    col("sender_id").alias("lookup_id"),
    col("timestamp_unix").alias("lookup_ts")
)

# Step 2: self-join to count how many times receiver was active as sender in past 1 hour
from pyspark.sql.functions import broadcast

receiver_outbound = df.alias("main").join(
    sender_activity.alias("lookup"),
    (col("main.receiver_id") == col("lookup.lookup_id")) &
    (col("lookup.lookup_ts") >= col("main.timestamp_unix") - 3600) &
    (col("lookup.lookup_ts") <= col("main.timestamp_unix")),
    "left"
).groupBy("main.txn_id").agg(
    count("lookup.lookup_id").alias("receiver_outbound_count_1h")
)

df = df.join(receiver_outbound, "txn_id", "left")
df = df.fillna({"receiver_outbound_count_1h": 0})

print("   ✓ Created receiver_inbound_count_1h, receiver_outbound_count_1h")

In [0]:
print("\n📊 Step 6: Engineering amount-based features...")

# Amount statistics per sender
sender_stats_window = Window.partitionBy("sender_id")

df = df.withColumn("sender_avg_amount", avg("amount_inr").over(sender_stats_window))
df = df.withColumn("sender_stddev_amount", stddev("amount_inr").over(sender_stats_window))

# Z-score calculation
df = df.withColumn(
    "amount_zscore",
    when(col("sender_stddev_amount") > 0,
         (col("amount_inr") - col("sender_avg_amount")) / col("sender_stddev_amount"))
    .otherwise(0)
)

# Amount percentile (global)
global_window = Window.orderBy("amount_inr")
df = df.withColumn("amount_percentile", percent_rank().over(global_window) * 100)

# Categorize z-score and percentile
df = df.withColumn(
    "amount_zscore_category",
    when(col("amount_zscore") > 3, 3)
    .when(col("amount_zscore") > 2, 2)
    .when(col("amount_zscore") > 1, 1)
    .otherwise(0)
)

df = df.withColumn(
    "amount_percentile_category",
    when(col("amount_percentile") > 95, 3)
    .when(col("amount_percentile") > 90, 2)
    .when(col("amount_percentile") > 75, 1)
    .otherwise(0)
)

print("   ✓ Created amount_zscore, amount_percentile, and categories")

In [0]:
print("\n📊 Step 7: Engineering temporal pattern features...")

# Weekend large transaction flag
df = df.withColumn(
    "weekend_large_txn",
    when((col("is_weekend") == True) & (col("amount_inr") > 15000), True)
    .otherwise(False)
)

# Merchant amount mismatch (if merchant transaction and amount unusual)
df = df.withColumn(
    "merchant_amount_mismatch",
    when((col("txn_type") == "P2M") & (col("amount_zscore") > 2), True)
    .otherwise(False)
)

print("   ✓ Created weekend_large_txn, merchant_amount_mismatch")

In [0]:
print("\n📊 Step 8: Checking failed transaction patterns...")

# Count failed attempts before current transaction
sender_failed_window = Window.partitionBy("sender_id").orderBy("timestamp_unix").rowsBetween(-5, -1)

df = df.withColumn(
    "failed_count_before",
    spark_sum(when(col("txn_status") == "FAILED", 1).otherwise(0)).over(sender_failed_window)
)

df = df.fillna({"failed_count_before": 0})

# Failed-then-success pattern
df = df.withColumn(
    "failed_then_success",
    when((col("failed_count_before") >= 2) & (col("txn_status") == "SUCCESS"), True)
    .otherwise(False)
)

print("   ✓ Created failed_count_before, failed_then_success")

In [0]:
print("\n📊 Step 9: Cleaning up and finalizing features...")

df = df.drop("timestamp_unix", "sender_avg_amount", "sender_stddev_amount")

# Fill nulls for safety
numeric_features = [
    "sender_txn_count_1min", "sender_txn_count_1hour", "sender_txn_count_24h",
    "time_since_last_txn_sec", "receiver_inbound_count_1h", "receiver_outbound_count_1h",
    "amount_zscore", "amount_percentile", "amount_zscore_category", "amount_percentile_category",
    "failed_count_before"
]

for col_name in numeric_features:
    df = df.fillna({col_name: 0})

boolean_features = [
    "sender_receiver_history", "weekend_large_txn", "merchant_amount_mismatch", 
    "failed_then_success"
]

for col_name in boolean_features:
    df = df.fillna({col_name: False})

print(f"✓ Final schema: {len(df.columns)} columns")
print(f"✓ Added {len(df.columns) - 35} new features")  # Approximate

In [0]:
print("\n📊 Step 10: Writing to Silver layer...")
print("   Target: workspace.silver.upi_features")
print("   Format: Delta Lake with optimizations")

try:
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("workspace.silver.upi_features")
    
    print("✅ Data successfully written to Delta Lake Silver layer!")
    
except Exception as e:
    print(f"⚠️  Unity Catalog write failed, trying DBFS...")
    dbfs_path = "dbfs:/suraksha/silver/upi_features"
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .save(dbfs_path)
    print(f"✅ Data written to: {dbfs_path}")

In [0]:
print("\n📊 Step 11: Verifying feature engineering...")

try:
    silver_df = spark.read.table("workspace.silver.upi_features")
    
    print(f"✓ Verified {silver_df.count():,} records in Silver table")
    print(f"✓ Total columns: {len(silver_df.columns)}")
    
    # Show key features for a sample transaction
    print("\n   Sample engineered features (first 3 rows):")
    feature_cols = [
        "txn_id", "sender_vpa", "amount_inr",
        "sender_txn_count_1min", "sender_txn_count_1hour",
        "time_since_last_txn_sec", "sender_receiver_history",
        "receiver_inbound_count_1h", "amount_zscore",
        "is_fraud", "fraud_type"
    ]
    silver_df.select(feature_cols).show(3, truncate=False)
    
except Exception as e:
    print(f"⚠️  Could not verify: {str(e)}")

In [0]:
print("\n" + "="*70)
print("✅ FEATURE ENGINEERING COMPLETE!")
print("="*70)
print("\n📊 Features Summary:")
print("  ✓ Velocity features (1min, 1hour, 24h windows)")
print("  ✓ Temporal features (time since last txn)")
print("  ✓ Historical features (sender-receiver history)")
print("  ✓ Mule detection features (inbound/outbound counts)")
print("  ✓ Statistical features (z-scores, percentiles)")
print("  ✓ Pattern features (weekend, merchant, failed-then-success)")

print("\n🎯 Data Architecture Progress:")
print("   Bronze Layer ✓ → workspace.bronze.upi (raw data)")
print("   Silver Layer ✓ → workspace.silver.upi_features (engineered features)")
print("   Gold Layer   ⏳ → Next: Model predictions")

print("\n✅ Ready for model training!")
print("   Next: Run 04_multiclass_training notebook")
print("\n" + "="*70)